In [ ]:
!pip install arcgis2geojson 

In [9]:
import requests
import json
from pathlib import Path
import geopandas as gpd
from shapely.geometry import shape

OUTPUT_DIR = Path("arcgis_geojson")
OUTPUT_DIR.mkdir(exist_ok=True)

SERVICES = {
    "states": "https://services.arcgis.com/P3ePLMYs2RVChkJx/ArcGIS/rest/services/USA_States_Generalized/FeatureServer/0/query?where=1=1&outFields=*&f=json",
    "counties": "https://services.arcgis.com/P3ePLMYs2RVChkJx/ArcGIS/rest/services/USA_Counties_Generalized/FeatureServer/0/query?where=1=1&outFields=*&f=geojson",
    "congressional_districts": "https://services.arcgis.com/P3ePLMYs2RVChkJx/ArcGIS/rest/services/CongressionalDistricts/FeatureServer/0/query?where=1=1&outFields=*&f=json",
    "tribal_lands": "https://services.arcgis.com/P3ePLMYs2RVChkJx/ArcGIS/rest/services/TribalLands/FeatureServer/0/query?where=1=1&outFields=*&f=json",
}

def esri_json_to_geojson(data):
    """Convert ESRI JSON to GeoJSON FeatureCollection."""
    features = []
    for feat in data.get("features", []):
        geom = shape(feat["geometry"])
        props = feat.get("attributes", {})
        features.append({"type": "Feature", "geometry": geom.__geo_interface__, "properties": props})
    return {"type": "FeatureCollection", "features": features}

def fetch_and_save(name, url):
    print(f"Downloading {name} …")
    r = requests.get(url)
    r.raise_for_status()

    out_file = OUTPUT_DIR / f"{name}.geojson"

    if url.endswith("f=geojson"):
        # Direct GeoJSON
        with open(out_file, "wb") as f:
            f.write(r.content)
        print(f"✅ Saved {name} (native GeoJSON) → {out_file}")
    else:
        # ESRI JSON → convert
        data = r.json()
        geojson = esri_json_to_geojson(data)
        with open(out_file, "w") as f:
            json.dump(geojson, f)
        print(f"✅ Saved {name} (converted from ESRI JSON) → {out_file}")

def main():
    for name, url in SERVICES.items():
        try:
            fetch_and_save(name, url)
        except Exception as e:
            print(f"❌ Failed to fetch {name}: {e}")

if __name__ == "__main__":
    main()


✅ Saved states (converted from ESRI JSON) → arcgis_geojson\states.geojson
✅ Saved counties (native GeoJSON) → arcgis_geojson\counties.geojson
✅ Saved congressional_districts (converted from ESRI JSON) → arcgis_geojson\congressional_districts.geojson
✅ Saved tribal_lands (converted from ESRI JSON) → arcgis_geojson\tribal_lands.geojson


In [10]:
import requests

url = "https://services.arcgis.com/P3ePLMYs2RVChkJx/ArcGIS/rest/services/USA_States_Generalized/FeatureServer/0/query?where=1=1&outFields=*&f=json"

r = requests.get(url)
print("Status:", r.status_code)
print("Content-Type:", r.headers.get("Content-Type", "unknown"))
print("Preview:\n", r.text[:500])  # just first 500 chars


Status: 200
Content-Type: application/json; charset=utf-8
Preview:
 {"error":{"code":499,"message":"Token Required","messageCode":"GWM_0003","details":["Token Required"]}}


In [2]:
import sys
print(sys.executable)


C:\Users\LSEng\AppData\Local\Programs\Python\Python311\python.exe


In [3]:
!{sys.executable} -m pip install arcgis2geojson


  Using cached arcgis2geojson-3.0.3-py3-none-any.whl.metadata (3.8 kB)
Using cached arcgis2geojson-3.0.3-py3-none-any.whl (6.1 kB)



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: C:\Users\LSEng\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip
